# N-Dimensional Tensor Autograd, VJPs & Reverse Broadcasting

In **Module 3**, we implemented reverse-mode autograd for scalar expressions. Now, we expand the engine to support **multi-dimensional tensors**, powering real deep learning workloads.

Scaling autograd to tensors requires solving two core mathematical challenges:

1. **Vector-Jacobian Products (VJP)**:
   For matrix operations like $Y = X @ W$, computing the explicit Jacobian matrix $J$ requires prohibitive $\mathcal{O}(M \cdot N)$ memory. Reverse-mode autograd evaluates the product of incoming gradients with the Jacobian directly:
   $$\frac{\partial L}{\partial X} = G @ W^T, \quad \frac{\partial L}{\partial W} = X^T @ G$$

2. **Automated Reverse Broadcasting (`unbroadcast`)**:
   During the forward pass, smaller tensors (e.g. bias vectors) are stretched across batch dimensions. In the backward pass, incoming gradients must be **summed along the broadcasted axes** to restore the original parameter shapes.

---

## Notebook Roadmap & Experiments

| # | Experiment | Mathematical Focus | Key Insight |
|:---:|:---|:---|:---|
| **01** | **Dense Layer Projection ($Y = X W + b$)** | Matrix VJP & Bias Unbroadcasting | Verifies that $W.\text{grad}$ matches $(D_{\text{in}}, D_{\text{out}})$ and bias gradients collapse the batch dimension via `unbroadcast`. |
| **02** | **Manual Calculus vs. Autograd (MSLE)** | Analytical equivalence | Implements Mean Squared Logarithmic Error to prove that one line of `loss.backward()` matches complex hand-derived NumPy vector derivatives.

In [1]:
from utorch import Tensor
import numpy as np

In [2]:
# Ground truth targets and predictions (Batch size N = 4)
y_true = Tensor([1.0, 2.0, 3.0, 4.0], name="y_true")
y_pred = Tensor([1.5, 2.5, 2.0, 5.0], name="y_pred")

# MSE Loss: (1 / N) * sum((y_pred - y_true) ** 2)
error = y_pred - y_true             # [+0.5, +0.5, -1.0, +1.0]
sq_err = error ** 2                 # [0.25, 0.25, 1.00, 1.00]
loss = sq_err.mean()                # 2.5 / 4 = 0.625
print("loss\t", loss)
loss.backward()

# Analytical gradient: 2/N * (y_pred - y_true) = (2 / 4) * error
expected_grad = 0.5 * error.value
print("Analytical loss.grad:\t", y_pred.grad.value)
print("Hand-calculated\t:    ", expected_grad)  # [0.25, 0.25, -0.5, 0.5]


loss	 ut,.Tensor(value=0.625, fun=mean)
Analytical loss.grad:	 [ 0.25  0.25 -0.5   0.5 ]
Hand-calculated	:     [ 0.25  0.25 -0.5   0.5 ]


In [6]:
# 1. Synthetic Dataset: Batch size B = 5, Feature dim D = 3
rng = np.random.default_rng(42)
X_data = np.abs(rng.standard_normal((5, 3)))
y_data = np.abs(rng.standard_normal((5, 1))) + 1.0

# Initial parameters
W_init = rng.standard_normal((3, 1))
b_init = np.array([0.5])


# =============================================================================
# APPROACH 1: The Tedious Manual Calculus (NumPy Lab Exercise)
# =============================================================================

W_manual = W_init.copy()
b_manual = b_init.copy()

# Forward
preds_manual = X_data @ W_manual + b_manual
msle_manual = np.mean((np.log(1.0 + y_data) - np.log(1.0 + preds_manual)) ** 2)

# Manual Backpropagation (Hand-derived chain rule):
# dL/d(preds) = (2 / B) * (ln(1 + preds) - ln(1 + y)) / (1 + preds)
B = X_data.shape[0]
d_preds = (2.0 / B) * (np.log(1.0 + preds_manual) - np.log(1.0 + y_data)) / (1.0 + preds_manual)

# dL/dW = X^T @ d_preds
grad_W_manual = X_data.T @ d_preds

# dL/db = sum(d_preds, axis=0) (unbroadcasting across batch)
grad_b_manual = np.sum(d_preds, axis=0)


# =============================================================================
# APPROACH 2: The Modern Way (utorch Automatic Differentiation)
# =============================================================================

# Convert to utorch Tensors
X = Tensor(X_data, name="X")
y = Tensor(y_data, name="y")
W = Tensor(W_init.copy(), name="W")
b = Tensor(b_init.copy(), name="b")

# Forward Pass: Exact same Python syntax
preds = (X @ W) + b

# MSLE Loss: L = mean( (log(1 + y) - log(1 + preds)) ** 2 )
loss = ((1.0 + y).log() - (1.0 + preds).log()) ** 2
loss = loss.mean()

# Backward Pass: One command!
loss.backward()


# =============================================================================
# VERIFICATION: Numerical Agreement
# =============================================================================

print(f"Manual MSLE Loss: {msle_manual:.6f}")
print(f"utorch MSLE Loss: {loss.value:.6f}")

print("\n--- Weight Gradients Comparison (dL/dW) ---")
print("Manual Calculus:\n", grad_W_manual)
print("utorch Autograd:\n", W.grad.value)
np.testing.assert_allclose(W.grad.value, grad_W_manual, rtol=1e-6)

print("\n--- Bias Gradients Comparison (dL/db) ---")
print("Manual Calculus: ", grad_b_manual)
print("utorch Autograd: ", b.grad.value)
np.testing.assert_allclose(b.grad.value, grad_b_manual, rtol=1e-6)


Manual MSLE Loss: 0.330723
utorch MSLE Loss: 0.330723

--- Weight Gradients Comparison (dL/dW) ---
Manual Calculus:
 [[-0.28436834]
 [-0.6875473 ]
 [-0.39877118]]
utorch Autograd:
 [[-0.28436834]
 [-0.6875473 ]
 [-0.39877118]]

--- Bias Gradients Comparison (dL/db) ---
Manual Calculus:  [-0.76224343]
utorch Autograd:  [-0.76224343]
